<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 03 · Independent Silver reference</h1><p>Goal: understand the expected rows before engine execution. This runs Python, not Spark or dbt. The lab is not complete until native outputs agree.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 03 · المرجع المستقل لـSilver</h1><p>الهدف فهم الصفوف المتوقعة قبل تشغيل المحرك. ينفذ هذا الدفتر Python وليس Spark أو dbt. لا يكتمل اللاب حتى تتطابق المخرجات الأصلية.</p></td></tr></tbody>
</table>



<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1 · Setup and immutable inputs</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١ · التجهيز والمدخلات الثابتة</h2></td></tr></tbody>
</table>



In [1]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset, new_workspace, write_json
from masar.silver_reference import (read_csv, drivers_index, normalize_trip, conformed_rows,
                                    reference_result, BUSINESS_FIELDS)
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day02_reference")
print("Input: MASAR_SMALL_V1 | Python reference only | Source files unchanged")

Input: MASAR_SMALL_V1 | Python reference only | Source files unchanged


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2 · Observe a value before and after normalization</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٢ · لاحظ القيمة قبل التوحيد وبعده</h2></td></tr></tbody>
</table>



In [2]:
base = read_csv(SOURCE, "trips.csv")
drivers = drivers_index(read_csv(SOURCE, "drivers.csv"))
raw = next(row for row in base if row["city"] != row["city"].strip().title())
clean, reasons = normalize_trip(raw, drivers)
print(json.dumps({"raw_city": raw["city"], "city": clean["city"],
                  "start_utc": clean["start_utc"], "duration_seconds": clean["duration_seconds"],
                  "reasons": reasons}, indent=2))

{
  "raw_city": " riyadh ",
  "city": "Riyadh",
  "start_utc": "2026-06-01T15:00:00Z",
  "duration_seconds": 1560,
  "reasons": []
}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3 · Two receipts are not two trips</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٣ · نسختا وصول لا تعنيان رحلتين</h2></td></tr></tbody>
</table>



In [3]:
initial, rejected = conformed_rows(base + base, drivers)
assert len(initial) == 72 and not rejected
print(json.dumps({"received_rows":len(base + base), "expected_business_rows":len(initial)}, indent=2))

{
  "received_rows": 144,
  "expected_business_rows": 72
}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4 · Midnight and the local start date</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٤ · منتصف الليل وتاريخ البداية المحلي</h2></td></tr></tbody>
</table>



In [4]:
late = read_csv(SOURCE, "late_trips.csv")
late_example, reasons = normalize_trip(late[0], drivers)
assert not reasons
print(json.dumps({key:late_example[key] for key in ("trip_id","start_utc","end_utc",
                  "trip_date_local","duration_seconds")},indent=2))

{
  "trip_id": "SYN_LATE001",
  "start_utc": "2026-06-01T20:55:00Z",
  "end_utc": "2026-06-01T21:08:00Z",
  "trip_date_local": "2026-06-01",
  "duration_seconds": 780
}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>5 · Compare the five reference scenarios</h2><p>The Bronze numbers are expected receipt counts, not observed Delta table counts.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٥ · قارن السيناريوهات المرجعية الخمسة</h2><p>أعداد Bronze توقعات لصفوف الاستقبال وليست أعداد جداول Delta مرصودة.</p></td></tr></tbody>
</table>



In [5]:
report = reference_result(SOURCE)
for step in report["stages"]:
    print(f"{step['stage']:24} Bronze expected={step['bronze_receipts']:3} | Silver expected={step['silver_rows']:2}")
print("Expected initial fare:",report["base_fare_sar"],"SAR")
print("Expected final fare:  ",report["final_fare_sar"],"SAR")
print("Late arrival changes business dates:",report["dates_after"])

base_and_replay          Bronze expected=144 | Silver expected=72
same_input_rerun         Bronze expected=144 | Silver expected=72
late_batch               Bronze expected=147 | Silver expected=75
same_batch_retry         Bronze expected=147 | Silver expected=75
intentional_redelivery   Bronze expected=150 | Silver expected=75
Expected initial fare: 1794.60 SAR
Expected final fare:   1875.60 SAR
Late arrival changes business dates: {'2026-06-01': 27, '2026-06-02': 24, '2026-06-03': 24}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>6 · Detect a join-grain trap</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٦ · اكشف خطأ مستوى الصف أثناء الربط</h2></td></tr></tbody>
</table>



In [6]:
print(json.dumps(report["join_demo"],indent=2))
assert report["join_demo"]["raw_gps_join_rows"] == 3 * report["join_demo"]["trip_rows"]

{
  "trip_rows": 72,
  "raw_gps_join_rows": 216,
  "note": "Raw one-to-many GPS joins change the grain; aggregate events before a trip-grain join."
}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>7 · Preserve checks and evidence</h2><p>Seven intentionally bad source rows are examined only in this isolated reference test, not ingested into Silver today.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٧ · احفظ الفحوص والأدلة</h2><p>تفحص صفوف المصدر السبعة المعيبة عمدًا في المرجع المعزول فقط، ولا تستقبل في Silver اليوم.</p></td></tr></tbody>
</table>



In [7]:
assert all(value is True for value in report["checks"].values())
for item in report["quality_examples"]:
    print(item["trip_id"] or "<missing trip_id>",item["reasons"])
print("Reference checks passed:",len(report["checks"]))
print("Engine executed:",report["engine_executed"],"| dbt executed:",report["dbt_executed"])
write_json(WORK / "day02_reference.json", report)

<missing trip_id> ['MISSING_TRIP_ID']
SYN_BAD002 ['INVALID_FARE']
SYN_BAD003 ['UNKNOWN_DRIVER']
SYN_BAD004 ['INVALID_TIMESTAMP']
SYN_BAD005 ['INVALID_DURATION']
SYN_BAD006 ['INVALID_DISTANCE']
SYN_BAD007 ['INVALID_CITY']
Reference checks passed: 16
Engine executed: False | dbt executed: False


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Next</h2><p>Continue to <a href="STUDENT.ipynb">native Lab 3a</a> using the successful Day 1 workspace. <a href="README.md">Day 2 guide</a>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التالي</h2><p>انتقل إلى <a href="STUDENT.ipynb">اللاب 3a الأصلي</a> باستخدام مساحة اليوم الأول الناجحة. <a href="README.md">دليل اليوم الثاني</a>.</p></td></tr></tbody>
</table>

